<style>
.markdown-content {
    direction: rtl;
    text-align: right;
}

.markdown-content pre {
    direction: ltr;
    text-align: left;
}

.markdown-content code {
    direction: ltr;
    text-align: left;
}
</style>

<div class="markdown-content">

# Working with PDF, TXT, Markdown, and HTML

## ۱. ایده‌ی اصلی

هر فرمت یک Loader مخصوص دارد؛ هدف: **حداکثر ساختار مفید، حداقل نویز** و خروجی یکسان `Document`.

## ۲. مقایسه‌ی فرمت‌ها

| فرمت | ساختار | خطر اصلی | کار لازم |
|---|---|---|---|
| TXT | ندارد | hard-wrap، encoding | چسباندن خطوط، حفظ پاراگراف |
| Markdown | عناوین، front matter | از دست رفتن سلسله‌مراتب | استخراج front matter و عناوین، نگه‌داشتن Markdown |
| HTML | تگ‌ها | نویز (nav, script, footer) | حذف نویز، تبدیل عناوین/لیست/جدول |
| PDF | ندارد (فقط مختصات) | ترتیب خوانش، اسکن، جدول | extractor مناسب، OCR، متادیتای صفحه |

## ۳. نکات Markdown

- Front matter ← metadata (`title`, `date`, ...)
- عناوین ← سلسله‌مراتب (برای metadata-aware chunking)
- `#` داخل code fence عنوان نیست
- Markdown رو به Markdown نگه دار

## ۴. نکات HTML

- حذف: `script, style, nav, footer, aside, form, noscript`
- حفظ: `h1..h6 → #`, `li → -`, جدول با `|`
- `<title>` جداگانه در metadata
- صفحات JS-rendered با parser ساده خالی درمیان (به‌جاش export/API)

## ۵. نکات PDF

| نوع | تشخیص | اقدام |
|---|---|---|
| Digital | متن معنادار | extractor معمولی |
| Scanned | متن خالی | OCR |
| ترکیبی | بعضی صفحات خالی | تشخیص صفحه‌به‌صفحه |

- یک Document به ازای هر صفحه، با `page` در metadata
- ابزارها: pypdf، pdfplumber، PyMuPDF (لایسنس AGPL)
- PDF فارسی: ترتیب معکوس، Presentation Forms (نیاز به NFKC)، `(cid:...)`، از دست رفتن نیم‌فاصله
- **قبل از انتخاب ابزار، روی PDF های واقعی bake-off کن**

## ۶. Quality Gate

| سنجه | علامت مشکل |
|---|---|
| متن خالی | اسکن یا فایل خراب |
| `\ufffd` | encoding اشتباه |
| `(cid:` | فونت بدون نگاشت |
| نسبت حروف پایین | متن زباله |

## ۷. الگوی production

**Fallback Chain:** extractor سریع ← Quality Gate ← extractor دقیق‌تر ← Quality Gate ← صف OCR/بررسی انسانی. هر فایل: extractor استفاده‌شده و flag ها لاگ بشه.

## ۸. ایده‌ی کد این نوت‌بوک

- `TextLoader` (با اصلاح hard-wrap اختیاری)
- `MarkdownLoader` (front matter + عناوین + نادیده‌گرفتن code fence)
- `HTMLLoader` با `html.parser` (حذف نویز، حفظ عنوان/لیست/جدول)
- `PDFLoader` با `pypdf` (یک Document به ازای هر صفحه، تشخیص PDF اسکن‌شده)
- ساخت یک PDF نمونه‌ی ساده بدون کتابخانه‌ی خارجی
- `quality_flags` و نمایش اثر `NFKC` روی Presentation Forms

## ۹. چک‌لیست production

- [ ] هر فرمت Loader جدا با خروجی یکسان دارد
- [ ] Markdown ساختارش (front matter، عناوین) حفظ شده
- [ ] HTML نویزگیری می‌شود و ساختار می‌ماند
- [ ] PDF: یک Document به ازای هر صفحه + `page`
- [ ] PDF اسکن‌شده تشخیص داده می‌شود (نه ایندکس خالی)
- [ ] Quality Gate بعد از هر Loader
- [ ] روی PDF های فارسی واقعی bake-off انجام شده
- [ ] extractor استفاده‌شده و flag ها لاگ می‌شوند

In [1]:
import hashlib
import re
import tempfile
import unicodedata
from dataclasses import dataclass, field
from datetime import datetime, timezone
from html.parser import HTMLParser
from pathlib import Path

try:
    from pypdf import PdfReader
    HAS_PYPDF = True
except ImportError:
    HAS_PYPDF = False


# ======================================================================
# 0) پایه: Document، رمزگشایی، metadata مشترک
# ======================================================================
@dataclass
class Document:
    doc_id: str
    text: str
    metadata: dict = field(default_factory=dict)


def decode_bytes(raw: bytes) -> tuple[str, str]:
    for enc in ("utf-8-sig", "cp1256"):
        try:
            return raw.decode(enc), enc
        except UnicodeDecodeError:
            continue
    return raw.decode("utf-8", errors="replace"), "utf-8(replace)"


def base_metadata(path: Path, raw: bytes, file_type: str) -> dict:
    return {
        "source": path.name,
        "file_type": file_type,
        "size_bytes": len(raw),
        "modified_at": datetime.fromtimestamp(path.stat().st_mtime, tz=timezone.utc).isoformat(),
        "content_hash": hashlib.sha256(raw).hexdigest(),
    }


def make_doc_id(path: Path, index: int = 0) -> str:
    return f"{hashlib.sha1(path.name.encode()).hexdigest()[:10]}-{index}"


# ======================================================================
# 1) TXT: رمزگشایی + اصلاح hard-wrap (اختیاری)
# ======================================================================
def unwrap_hard_lines(text: str) -> str:
    """خطوط شکسته‌شده‌ی پشت‌سرهم را می‌چسباند؛ مرز پاراگراف و bullet ها حفظ می‌شود."""
    out, buffer = [], []
    for line in text.replace("\r\n", "\n").split("\n"):
        stripped = line.strip()
        if not stripped:                                  # خط خالی = مرز پاراگراف
            if buffer:
                out.append(" ".join(buffer)); buffer = []
            out.append("")
        elif re.match(r"^([-*•]|\d+[.)])\s", stripped):   # bullet شروع آیتم جدید است
            if buffer:
                out.append(" ".join(buffer)); buffer = []
            buffer.append(stripped)
        else:
            buffer.append(stripped)
    if buffer:
        out.append(" ".join(buffer))
    return "\n".join(out)


class TextLoader:
    def __init__(self, unwrap: bool = False):
        self.unwrap = unwrap

    def load(self, path: Path) -> list[Document]:
        raw = path.read_bytes()
        text, enc = decode_bytes(raw)
        if self.unwrap:
            text = unwrap_hard_lines(text)
        meta = {**base_metadata(path, raw, "txt"), "encoding": enc}
        return [Document(make_doc_id(path), text, meta)]


# ======================================================================
# 2) Markdown: front matter + عناوین (بدون توجه به code fence)
# ======================================================================
FRONT_MATTER = re.compile(r"\A---\n(.*?)\n---\n", re.S)


class MarkdownLoader:
    def load(self, path: Path) -> list[Document]:
        raw = path.read_bytes()
        text, enc = decode_bytes(raw)
        text = text.replace("\r\n", "\n")

        front = {}
        m = FRONT_MATTER.match(text)
        if m:
            for line in m.group(1).splitlines():
                if ":" in line:
                    k, v = line.split(":", 1)
                    front[k.strip()] = v.strip()
            text = text[m.end():]

        text = re.sub(r"<!--.*?-->", "", text, flags=re.S).strip()

        headings, in_code = [], False
        for line in text.splitlines():
            if line.strip().startswith("```"):
                in_code = not in_code
                continue
            if not in_code:
                hm = re.match(r"^(#{1,6})\s+(.*)$", line)
                if hm:
                    headings.append({"level": len(hm.group(1)), "title": hm.group(2).strip()})

        meta = {**base_metadata(path, raw, "md"), "encoding": enc,
                "front_matter": front, "headings": headings,
                "title": front.get("title") or (headings[0]["title"] if headings else path.stem)}
        return [Document(make_doc_id(path), text, meta)]


# ======================================================================
# 3) HTML: حذف نویز + حفظ ساختار (عنوان، لیست، جدول)
# ======================================================================
class _HTMLExtractor(HTMLParser):
    SKIP = {"script", "style", "nav", "footer", "aside", "form", "noscript"}
    BLOCK = {"p", "div", "section", "article", "br", "tr", "ul", "ol", "table"}

    def __init__(self):
        super().__init__(convert_charrefs=True)
        self.parts, self.title = [], ""
        self._skip_depth, self._in_title = 0, False

    def handle_starttag(self, tag, attrs):
        if tag in self.SKIP:
            self._skip_depth += 1
            return
        if self._skip_depth:
            return
        if tag == "title":
            self._in_title = True
        elif re.fullmatch(r"h[1-6]", tag):
            self.parts.append("\n\n" + "#" * int(tag[1]) + " ")
        elif tag == "li":
            self.parts.append("\n- ")
        elif tag in ("td", "th"):
            self.parts.append(" | ")
        elif tag in self.BLOCK:
            self.parts.append("\n")

    def handle_endtag(self, tag):
        if tag in self.SKIP:
            self._skip_depth = max(0, self._skip_depth - 1)
            return
        if self._skip_depth:
            return
        if tag == "title":
            self._in_title = False
        elif re.fullmatch(r"h[1-6]", tag) or tag == "p":
            self.parts.append("\n")

    def handle_data(self, data):
        if self._skip_depth:
            return
        if self._in_title:
            self.title += data
        else:
            self.parts.append(data)


class HTMLLoader:
    def load(self, path: Path) -> list[Document]:
        raw = path.read_bytes()
        html, enc = decode_bytes(raw)
        ex = _HTMLExtractor()
        ex.feed(html)
        text = "".join(ex.parts)
        text = re.sub(r"[ \t]+", " ", text)
        text = "\n".join(line.strip() for line in text.splitlines())
        text = re.sub(r"\n{3,}", "\n\n", text).strip()
        meta = {**base_metadata(path, raw, "html"), "encoding": enc,
                "title": ex.title.strip() or path.stem}
        return [Document(make_doc_id(path), text, meta)]


# ======================================================================
# 4) PDF: یک Document به ازای هر صفحه + تشخیص PDF اسکن‌شده
# ======================================================================
class NeedsOCR(Exception):
    pass


class PDFLoader:
    def load(self, path: Path) -> list[Document]:
        if not HAS_PYPDF:
            raise RuntimeError("pypdf نصب نیست: pip install pypdf")
        raw = path.read_bytes()
        reader = PdfReader(path)
        if reader.is_encrypted:
            raise RuntimeError("PDF رمزگذاری شده است")

        base = base_metadata(path, raw, "pdf")
        docs = []
        for i, page in enumerate(reader.pages):
            text = (page.extract_text() or "").strip()
            if text:
                docs.append(Document(make_doc_id(path, i), text,
                                     {**base, "page": i + 1, "total_pages": len(reader.pages),
                                      "extractor": "pypdf"}))
        if not docs:
            raise NeedsOCR(f"{path.name}: هیچ متنی استخراج نشد (احتمالاً اسکن‌شده)")
        return docs


# ======================================================================
# 5) Quality Gate
# ======================================================================
def quality_flags(doc: Document) -> list[str]:
    text, flags = doc.text, []
    if not text.strip():
        flags.append("empty")
    if "\ufffd" in text:
        flags.append("replacement_chars")
    if "(cid:" in text:
        flags.append("cid_garbage")
    non_space = [c for c in text if not c.isspace()]
    if non_space and sum(c.isalpha() for c in non_space) / len(non_space) < 0.6:
        flags.append("low_letter_ratio")
    return flags


# ======================================================================
# 6) ساخت PDF نمونه بدون کتابخانه‌ی خارجی (فقط متن لاتین، فونت Helvetica)
# ======================================================================
def make_pdf(pages: list[str]) -> bytes:
    def esc(s: str) -> str:
        return s.replace("\\", "\\\\").replace("(", "\\(").replace(")", "\\)")

    n = len(pages)
    page_ids = [4 + 2 * i for i in range(n)]
    content_ids = [5 + 2 * i for i in range(n)]
    objs = [b"<< /Type /Catalog /Pages 2 0 R >>"]
    kids = " ".join(f"{p} 0 R" for p in page_ids)
    objs.append(f"<< /Type /Pages /Kids [{kids}] /Count {n} >>".encode())
    objs.append(b"<< /Type /Font /Subtype /Type1 /BaseFont /Helvetica >>")
    for cid, text in zip(content_ids, pages):
        objs.append((f"<< /Type /Page /Parent 2 0 R /MediaBox [0 0 612 792] /Contents {cid} 0 R "
                     f"/Resources << /Font << /F1 3 0 R >> >> >>").encode())
        lines = [l for l in text.split("\n") if l]
        body = "BT /F1 12 Tf 72 720 Td 16 TL " + " T* ".join(f"({esc(l)}) Tj" for l in lines) + " ET"
        stream = body.encode("latin-1")
        objs.append(b"<< /Length %d >>\nstream\n" % len(stream) + stream + b"\nendstream")

    out, offsets = bytearray(b"%PDF-1.4\n"), []
    for i, body in enumerate(objs, 1):
        offsets.append(len(out))
        out += f"{i} 0 obj\n".encode() + body + b"\nendobj\n"
    xref = len(out)
    out += f"xref\n0 {len(objs) + 1}\n".encode() + b"0000000000 65535 f \n"
    for off in offsets:
        out += f"{off:010d} 00000 n \n".encode()
    out += (f"trailer\n<< /Size {len(objs) + 1} /Root 1 0 R >>\nstartxref\n{xref}\n%%EOF\n").encode()
    return bytes(out)


# ======================================================================
# 7) اجرا
# ======================================================================
LOADERS = {".txt": TextLoader(unwrap=True), ".md": MarkdownLoader(),
           ".html": HTMLLoader(), ".htm": HTMLLoader(), ".pdf": PDFLoader()}

MD_SAMPLE = """---
title: آیین‌نامه‌ی مرخصی
version: 2025
---
# مرخصی
## مرخصی استحقاقی
هر کارمند ۳۰ روز مرخصی دارد.
```python
# این یک کامنت است، نه عنوان
x = 1
```
## مرخصی بدون حقوق
حداکثر ۳۰ روز در سال.
"""

HTML_SAMPLE = """<html><head><title>سؤالات متداول</title><style>body{color:red}</style></head>
<body>
<nav>خانه | درباره ما | تماس</nav>
<h1>سؤالات متداول</h1>
<p>پاسخ به پرسش‌های رایج کارمندان.</p>
<h2>بیمه</h2>
<ul><li>درمان</li><li>دندانپزشکی</li></ul>
<table><tr><th>نوع</th><th>سقف</th></tr><tr><td>بستری</td><td>۱۰۰ میلیون</td></tr></table>
<script>alert('x')</script>
<footer>کلیه حقوق محفوظ است</footer>
</body></html>"""

TXT_SAMPLE = ("مرخصی استحقاقی هر کارمند سی روز در سال است و باید با تأیید\n"
              "مدیر مستقیم ثبت شود.\n\n"
              "- مرخصی بدون حقوق نیاز به تأیید منابع انسانی دارد\n"
              "- مرخصی استعلاجی با گواهی پزشک است\n")

with tempfile.TemporaryDirectory() as tmp:
    root = Path(tmp)
    (root / "leave.md").write_text(MD_SAMPLE, encoding="utf-8")
    (root / "faq.html").write_text(HTML_SAMPLE, encoding="utf-8")
    (root / "notes.txt").write_text(TXT_SAMPLE, encoding="utf-8")
    (root / "policy.pdf").write_bytes(make_pdf(["Leave Policy\nAnnual leave: 30 days", "Unpaid leave: max 30 days"]))
    (root / "scanned.pdf").write_bytes(make_pdf(["", ""]))      # PDF بدون متن (شبیه اسکن)

    for name in ["notes.txt", "leave.md", "faq.html", "policy.pdf", "scanned.pdf"]:
        path = root / name
        print("=" * 60, f"\n{name}\n" + "=" * 60)
        try:
            docs = LOADERS[path.suffix].load(path)
        except NeedsOCR as e:
            print(f"  ⚠️ NeedsOCR → به صف OCR می‌رود: {e}")
            continue
        except RuntimeError as e:
            print(f"  ⏭️ رد شد: {e}")
            continue

        for d in docs:
            m = d.metadata
            extra = {k: m[k] for k in ("title", "page", "headings", "front_matter") if k in m}
            print(f"  [{d.doc_id}] flags={quality_flags(d) or '✅'} | {extra}")
            print("  ---")
            print("  " + d.text.replace("\n", "\n  "))

    print("\n" + "=" * 60, "\nQuality Gate روی متن‌های خراب\n" + "=" * 60)
    bad = {
        "encoding اشتباه": "مرخصی \ufffd\ufffd روز",
        "فونت بدون نگاشت": "(cid:12)(cid:45)(cid:7)(cid:99)",
        "نمادهای زباله": "### $$$ %%% ^^^ &&& ***",
    }
    for label, text in bad.items():
        print(f"  {label:<16} → {quality_flags(Document('x', text))}")

    print("\n" + "=" * 60, "\nPresentation Forms و NFKC (مقدمه‌ی تاپیک بعدی)\n" + "=" * 60)
    for label, s in {"alef منفرد (U+FE8D)": "\ufe8d", "لام-الف مد (U+FEF5)": "\ufef5"}.items():
        norm = unicodedata.normalize("NFKC", s)
        print(f"  {label}: {[hex(ord(c)) for c in s]} → {[hex(ord(c)) for c in norm]}")

notes.txt
  [83348d283c-0] flags=✅ | {}
  ---
  مرخصی استحقاقی هر کارمند سی روز در سال است و باید با تأیید مدیر مستقیم ثبت شود.
  
  - مرخصی بدون حقوق نیاز به تأیید منابع انسانی دارد
  - مرخصی استعلاجی با گواهی پزشک است
  
leave.md
  [34004321a3-0] flags=✅ | {'title': 'آیین\u200cنامه\u200cی مرخصی', 'headings': [{'level': 1, 'title': 'مرخصی'}, {'level': 2, 'title': 'مرخصی استحقاقی'}, {'level': 2, 'title': 'مرخصی بدون حقوق'}], 'front_matter': {'title': 'آیین\u200cنامه\u200cی مرخصی', 'version': '2025'}}
  ---
  # مرخصی
  ## مرخصی استحقاقی
  هر کارمند ۳۰ روز مرخصی دارد.
  ```python
  # این یک کامنت است، نه عنوان
  x = 1
  ```
  ## مرخصی بدون حقوق
  حداکثر ۳۰ روز در سال.
faq.html
  [64c538dd92-0] flags=✅ | {'title': 'سؤالات متداول'}
  ---
  # سؤالات متداول
  
  پاسخ به پرسش‌های رایج کارمندان.
  
  ## بیمه
  
  - درمان
  - دندانپزشکی
  
  | نوع | سقف
  | بستری | ۱۰۰ میلیون
policy.pdf
  ⏭️ رد شد: pypdf نصب نیست: pip install pypdf
scanned.pdf
  ⏭️ رد شد: pypdf نصب نیست: pip install pypdf

Qual